# 07 — Computational Cost Evaluation

This includes index build time, retrieval latency, embedding/scoring time, number of documents scored, similarity calculations, peak RSS, iterations to termination, LLM calls, and estimated LLM tokens. Retrieval overhead is computed as (SAR retrieval latency - baseline retrieval latency) / baseline retrieval latency * 100 when a baseline run directory is supplied; it is not treated as the full system cost when methods differ in the number of MIGRES iterations they require.

In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text())
BASE_CONFIG = [ROOT / "configs" / "default.yaml"]


In [ ]:
RUN_EXPERIMENT=False
METHOD='sar'
DATASET='hotpotqa'
LIMIT=20
RUN_ROOT=ROOT/'outputs/experiments/07_computational_cost'
BASELINE_RUN_DIR=None  # optional completed baseline run for retrieval-overhead comparison


In [ ]:
from sar_migres.config import load_config
from sar_migres.datasets import load_phase_questions
from sar_migres.evaluation import retrieval_overhead_pct
from sar_migres.experiment_runner import baseline_overrides,cost_metrics_from_run,frozen_setting
from sar_migres.pipeline import SARMigresPipeline
base=load_config(*BASE_CONFIG)
k=base.sar.k
fp=ROOT/'outputs/experiments/01_parameter_selection/selected_config.json'
if fp.exists(): k,_=frozen_setting(fp)
if RUN_EXPERIMENT:
    ov={'retrieval':{'method':METHOD},'sar':{'k':k},'baseline':{'k':k}} if METHOD=='sar' else baseline_overrides(METHOD,k=k,run_name=f'cost_{DATASET}_{METHOD}')
    ov['data']={'dataset':DATASET,'dataset_path':base.data.dataset_paths[DATASET],'split':base.data.final_split}
    ov['run']={'name':f'cost_{DATASET}_{METHOD}','output_dir':str(RUN_ROOT),'resume':False}
    items=load_phase_questions(DATASET,base.data.dataset_paths[DATASET],'final',limit=LIMIT,final_split=base.data.final_split,validation_fraction=base.data.validation_fraction,validation_seed=base.data.validation_seed)
    pipe=SARMigresPipeline(load_config(*BASE_CONFIG,overrides=ov),run_id=f'cost_{DATASET}_{METHOD}').prepare()
    pipe.run_batch(items,resume=False)
    sar_cost=cost_metrics_from_run(pipe.run_dir)
    if BASELINE_RUN_DIR:
        baseline_cost=cost_metrics_from_run(BASELINE_RUN_DIR)
        sar_cost['retrieval_overhead_pct_vs_baseline']=retrieval_overhead_pct(sar_cost['retrieval_latency_total_s'],baseline_cost['retrieval_latency_total_s'])
        print('Baseline cost:',json.dumps(baseline_cost,indent=2,allow_nan=True))
    print(json.dumps(sar_cost,indent=2,allow_nan=True))
else:
    print('Preflight only. Method:',METHOD,'Dataset:',DATASET,'K:',k)
